Import Library

In [ ]:
import os
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

Path Dataset

In [ ]:
RAW_PATH = Path("../data/raw")
OUTPUT_PATH = Path("../data/processed")

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

Helper Function

In [ ]:
import re
import numpy as np

def clean_number(value):
    """
    Convert Indonesian marketplace numbers.

    Examples:
    ----------
    107.000   -> 107000
    Rp107.000 -> 107000
    596,4k    -> 596400
    1,6m      -> 1600000
    1RB+      -> 1000
    2,5RB+    -> 2500
    15RB      -> 15000
    1JT+      -> 1000000
    2,5JT     -> 2500000
    100%      -> 100
    """

    if value is None:
        return np.nan

    if isinstance(value, (int, float)):
        return value

    value = str(value).strip().lower()

    if value == "":
        return np.nan

    value = value.replace("rp", "")
    value = value.replace("%", "")
    value = value.replace("+", "")
    value = value.replace(" ", "")

    # Shopee Indonesia
    if "rb" in value:
        number = value.replace("rb", "").replace(",", ".")
        return float(number) * 1000

    if "jt" in value:
        number = value.replace("jt", "").replace(",", ".")
        return float(number) * 1000000

    # Format internasional
    if "k" in value:
        number = value.replace("k", "").replace(",", ".")
        return float(number) * 1000

    if "m" in value:
        number = value.replace("m", "").replace(",", ".")
        return float(number) * 1000000

    # Harga Indonesia
    value = value.replace(".", "")
    value = value.replace(",", ".")

    try:
        return float(value)

    except:
        return np.nan

Seller Joined

In [ ]:
def extract_joined_year(value):

    if value is None:
        return np.nan

    match = re.search(r'(\d+)', str(value))

    if match:
        return int(match.group(1))

    return np.nan

Response Time

In [ ]:
def encode_response_time(value):

    if value is None:
        return np.nan

    value = value.lower()

    if "minutes" in value:
        return 1

    if "hours" in value:
        return 2

    if "days" in value:
        return 3

    return np.nan

Read All JSON

In [ ]:
records = []

for file in RAW_PATH.glob("*.json"):

    keyword = file.stem

    print(f"Loading {file.name}")

    with open(file, "r", encoding="utf-8") as f:
        raw = json.load(f)

    for item in raw["data"]:

        record = {

            "keyword": keyword,

            "product_id": item.get("id"),

            "product_name": item.get("name"),

            "url": item.get("url"),

            "currency": item["price"].get("currency"),

            "original_price_min": item["price"].get("original_price_min"),

            "original_price_max": item["price"].get("original_price_max"),

            "final_price_min": item["price"].get("final_price_min"),

            "final_price_max": item["price"].get("final_price_max"),

            "discount_percent": item["price"].get("discount_percent"),

            "category_1": item["categoryPath"][1] if len(item["categoryPath"]) > 1 else None,

            "category_2": item["categoryPath"][2] if len(item["categoryPath"]) > 2 else None,

            "category_3": item["categoryPath"][3] if len(item["categoryPath"]) > 3 else None,

            "rating": item["rating"].get("average"),

            "review_count": item["rating"].get("reviewCount"),

            "sold": item["rating"].get("sold"),

            "star_1": item["rating"]["starRating"].get("1"),

            "star_2": item["rating"]["starRating"].get("2"),

            "star_3": item["rating"]["starRating"].get("3"),

            "star_4": item["rating"]["starRating"].get("4"),

            "star_5": item["rating"]["starRating"].get("5"),

            "seller_id": item["seller"].get("id"),

            "seller_name": item["seller"].get("name"),

            "seller_rating": item["seller"].get("rating"),

            "seller_followers": item["seller"].get("follower"),

            "seller_product": item["seller"].get("product"),

            "seller_response_rate": item["seller"].get("responseRate"),

            "seller_response_time": item["seller"].get("responseTime"),

            "seller_joined": item["seller"].get("joined"),

        }

        records.append(record)

df = pd.DataFrame(records)

Preview Dataset

In [ ]:
print(df.shape)

df.head()

Cleaning Numeric Data

In [ ]:
numeric_columns = [

    "original_price_min",

    "original_price_max",

    "final_price_min",

    "final_price_max",

    "discount_percent",

    "rating",

    "review_count",

    "sold",

    "star_1",

    "star_2",

    "star_3",

    "star_4",

    "star_5",

    "seller_rating",

    "seller_followers",

    "seller_product",

    "seller_response_rate"

]

for col in numeric_columns:
    df[col] = df[col].apply(clean_number)

Feature Engineering

In [ ]:
df["price_avg"] = (
    df["final_price_min"] +
    df["final_price_max"]
) / 2

df["price_range"] = (
    df["final_price_max"] -
    df["final_price_min"]
)

df["seller_age"] = df["seller_joined"].apply(extract_joined_year)

df["seller_response_time"] = df["seller_response_time"].apply(
    encode_response_time
)

Missing Value

In [ ]:
missing = pd.DataFrame({

    "Missing": df.isnull().sum(),

    "Percent": (df.isnull().sum()/len(df))*100

})

missing.sort_values("Percent", ascending=False)

Duplicate

In [ ]:
print("Duplicate :", df.duplicated().sum())

df = df.drop_duplicates()

Data Info

In [ ]:
df.info()

Statistik

In [ ]:
df.describe(include="all")

Save CSV

In [ ]:
output = OUTPUT_PATH / "ecommerce.csv"

df.to_csv(output, index=False, encoding="utf-8-sig")

print(output)

Final Preview

In [ ]:
df.head()